# nb01 — LiteLLM Proxy 연결 확인

> **목적**: 강사 노트북에서 실행 중인 LiteLLM proxy에 접속해 Gemini API를 사용할 수 있는지 확인한다.  
> **소요시간**: 약 20분  
> **사전 조건**: 강사 노트북과 같은 와이파이에 연결되어 있어야 한다.

---

## 오늘 실습의 네트워크 구조

```
수강생 노트북 (여러분)
       ↓  같은 와이파이
 강사 노트북 :4000  ← LiteLLM proxy
       ↓
 Gemini API  ← 강사 API 키로 호출
```

개인 API 키 없이도 Gemini를 사용할 수 있다. 모든 요청은 강사 노트북을 경유한다.

## Step 0. 강사 Proxy 주소 설정

> 🔴 **강사가 공지한 IP 주소**를 아래에 입력하세요.

화면에 표시된 IP 주소를 `INSTRUCTOR_IP` 변수에 넣어주세요.

In [ ]:
# ✏️ 강사가 공지한 IP 주소로 변경하세요
INSTRUCTOR_IP = "192.168.0.100"   # ← 여기를 수정

# 연결 정보 조립
LITELLM_BASE_URL = f"http://{INSTRUCTOR_IP}:4000/v1"
LITELLM_API_KEY  = "sk-workshop-2025"
MODEL_NAME       = "gemini-2.5-flash-lite"

print(f"proxy URL : {LITELLM_BASE_URL}")
print(f"API key   : {LITELLM_API_KEY}")
print(f"model     : {MODEL_NAME}")

## Step 1. 패키지 설치 및 클라이언트 준비

LiteLLM proxy는 **OpenAI 호환 API**를 제공한다.  
따라서 `openai` 패키지를 그대로 사용하되, 접속 주소만 강사 proxy로 바꿔준다.

In [ ]:
# openai 패키지 설치 (이미 설치된 경우 무시됨)
!pip install openai --quiet

In [ ]:
from openai import OpenAI

# 강사 proxy에 연결하는 클라이언트 생성
client = OpenAI(
    base_url=LITELLM_BASE_URL,
    api_key=LITELLM_API_KEY
)

print("✅ OpenAI 클라이언트 생성 완료")

## Step 2. 연결 테스트 — API 호출

간단한 메시지를 보내서 응답이 오는지 확인한다.  
응답이 돌아오면 proxy ↔ Gemini 연결이 정상이다.

In [ ]:
# 기본 연결 테스트
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "한 문장으로 자기소개를 해줘."}
    ],
    max_tokens=100
)

print("📨 모델 응답:")
print(response.choices[0].message.content)
print()
print(f"사용 모델: {response.model}")
print(f"입력 토큰: {response.usage.prompt_tokens}")
print(f"출력 토큰: {response.usage.completion_tokens}")

### ✅ 정상 출력 예시

```
📨 모델 응답:
안녕하세요! 저는 Google이 개발한 AI 어시스턴트 Gemini입니다.

사용 모델: gemini-2.5-flash-lite
입력 토큰: 12
출력 토큰: 24
```

위와 비슷한 출력이 나오면 **연결 성공**입니다.

In [ ]:
# 시스템 프롬프트 + 사용자 메시지 테스트 (이후 실습에서 쓰는 패턴 미리 연습)
response2 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "당신은 보안 전문가입니다. 항상 한국어로 답하세요."},
        {"role": "user",   "content": "LLM 보안에서 가장 위험한 공격 유형 하나만 알려줘."}
    ],
    max_tokens=150
)

print("📨 시스템 프롬프트 포함 응답:")
print(response2.choices[0].message.content)

## Step 3. 연결 체크리스트

오류가 발생했다면 아래 표를 참고해 원인을 찾아보세요.

| 오류 메시지 | 원인 | 해결 방법 |
|---|---|---|
| `Connection refused` | IP 주소 오류 또는 proxy 미실행 | IP 재확인 후 `INSTRUCTOR_IP` 수정 |
| `ConnectTimeout` | 와이파이 불일치 | 강사와 동일한 와이파이에 연결 확인 |
| `401 Unauthorized` | API key 오류 | `sk-workshop-2025` 정확히 입력되었는지 확인 |
| `404 Not Found` | URL 경로 오류 | `base_url` 끝에 `/v1` 포함 확인 |
| `429 Too Many Requests` | Rate limit 초과 | 잠시 기다린 후 재시도 |
| `model not found` | 모델명 오류 | `gemini-2.5-flash-lite` 정확히 입력 |

In [ ]:
# 진단 셀 — 오류 원인 파악용
import socket
import urllib.request

print("=== 연결 진단 ===")

# 1. 와이파이 IP 확인
hostname = socket.gethostname()
local_ip = socket.gethostbyname(hostname)
print(f"내 노트북 IP: {local_ip}")
print(f"강사 proxy IP: {INSTRUCTOR_IP}")

# 2. HTTP 연결 확인
proxy_url = f"http://{INSTRUCTOR_IP}:4000"
try:
    urllib.request.urlopen(proxy_url, timeout=5)
    print(f"\n✅ {proxy_url} 접속 가능")
except Exception as e:
    print(f"\n❌ {proxy_url} 접속 실패: {e}")
    print("   → IP 주소를 다시 확인하거나 강사에게 문의하세요")

## 완료 확인

아래 두 가지가 모두 성공했으면 nb01 실습 완료입니다.

- [ ] Step 2: 모델 응답이 정상적으로 출력됨
- [ ] Step 2: 시스템 프롬프트 포함 응답도 정상 출력됨

---

## 다음 실습

> **nb02 — Gandalf CTF**  
> 브라우저에서 AI의 비밀번호를 알아내는 프롬프트 인젝션 게임을 직접 체험합니다.  
> 접속 주소: [https://gandalf.lakera.ai](https://gandalf.lakera.ai)